In [1]:
import os
import time
import shutil
from pathlib import Path
import re

import pandas as pd
from tqdm import tqdm
from IPython.display import Audio as IPythonAudio, display
from IPython.display import Markdown, display as ipy_display
from mutagen.mp3 import MP3
from mutagen.wave import WAVE
import torch
import torchaudio
from concurrent.futures import ThreadPoolExecutor, as_completed

from datasets import Dataset, Audio
from datasets import load_dataset

from utils import download_audios, download_texts
from utils import usx_parser
from utils import audio_stats
from utils import data_checks
from utils import speaker_identifier
# from utils import force_align_book

/home/mila/g/guzmand/scratch/.conda/envs/ReadAlongs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mila/g/guzmand/scratch/.conda/envs/ReadAlongs/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [6]:
LANGUAGE = "Malayalam"

# Download

## Audios and timing files

In [9]:
html_path = Path(f"html_files/audio/{LANGUAGE}.html")
# 1) Parse HTML -> dict(name -> URL)
links = download_audios.extract_artifact_links(str(html_path))
links

{'Genesis': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f6d',
  'section': 'Old Testament - mp3'},
 'Exodus': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f9a',
  'section': 'Old Testament - mp3'},
 'Leviticus': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f97',
  'section': 'Old Testament - mp3'},
 'Numbers': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f89',
  'section': 'Old Testament - mp3'},
 'Deuteronomy': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f91',
  'section': 'Old Testament - mp3'},
 'Joshua': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f6b',
  'section': 'Old Testament - mp3'},
 'Judges': {'url': 'https://openbible-api-1.biblica.com/artifactContent/646bcf44e901194107c33f77',
  'section': 'Old Testament - mp3'},
 'Ruth': {'url': 'https://openbible-ap

In [11]:
# Directory containing HTML files
html_dir = Path("html_files/audio")
output_base_dir = Path("data/audios")

# Get all HTML files
html_files = sorted(html_dir.glob("*.html"))
html_files = [f for f in html_files if LANGUAGE in f.stem]

print(f"Found {len(html_files)} languages to process")

for i, html_path in enumerate(html_files):
    lang_name = html_path.stem  # Get filename without extension
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(html_files)}] Processing: {lang_name}")
    print(f"{'='*60}")
    
    try:
        # 1) Parse HTML -> dict(name -> URL)
        links = download_audios.extract_artifact_links(str(html_path))
        print(f"Found {len(links)} artifacts")
        
        # 2) Download + unzip into folders
        output_dir = output_base_dir / lang_name
        download_audios.download_and_unzip_all(links, str(output_dir), overwrite=False, timeout=60)
        
        print(f"✓ Successfully processed {lang_name}")
        
    except Exception as e:
        print(f"✗ Failed to process {lang_name}: {e}")
        continue
    
    # Sleep 1 minute between languages (except after the last one)
    if i < len(html_files) - 1:
        print(f"\nSleeping for 1 minute before next language...")
        time.sleep(60)

print(f"\n{'='*60}")
print("All languages processed!")
print(f"{'='*60}")

Found 1 languages to process

[1/1] Processing: Malayalam
Found 67 artifacts


Processing books:   0%|          | 0/67 [00:00<?, ?book/s, Old Testament - mp3/Genesis]

Processing books: 100%|██████████| 67/67 [04:11<00:00,  3.75s/book, Timing Files/Timing Files Bundle]   

✓ Successfully processed Malayalam

All languages processed!


## Text files

In [12]:
html_path = Path(f"html_files/text/{LANGUAGE}.html")
# 1) Parse HTML -> dict(name -> URL)
links = download_texts.extract_artifact_links(str(html_path))
links

{'USX': 'https://openbible-api-1.biblica.com/artifactContent/6631d010c8d9e67f006e7079',
 'PARATEXT (USFM)': 'https://openbible-api-1.biblica.com/artifactContent/6631d2d1c8d9e67f006e708c'}

In [13]:
# Directory containing HTML files
html_dir = Path("html_files/text")
output_base_dir = Path("data/texts")

# Get all HTML files
html_files = sorted(html_dir.glob("*.html"))
html_files = [f for f in html_files if LANGUAGE in f.stem]

print(f"Found {len(html_files)} languages to process")

for i, html_path in enumerate(html_files):
    lang_name = html_path.stem  # Get filename without extension
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(html_files)}] Processing: {lang_name}")
    print(f"{'='*60}")
    
    try:
        # 1) Parse HTML -> dict(name -> URL)
        links = download_texts.extract_artifact_links(str(html_path))
        print(f"Found {len(links)} artifacts")
        
        # 2) Download + unzip into folders
        output_dir = output_base_dir / lang_name
        download_texts.download_and_unzip_all(links, str(output_dir), overwrite=False, timeout=60)
        
        print(f"✓ Successfully processed {lang_name}")
        
    except Exception as e:
        print(f"✗ Failed to process {lang_name}: {e}")
        continue
    
    # Sleep 1 minute between languages (except after the last one)
    if i < len(html_files) - 1:
        print(f"\nSleeping for 1 minute before next language...")
        # time.sleep(60)

print(f"\n{'='*60}")
print("All languages processed!")
print(f"{'='*60}")

Found 1 languages to process

[1/1] Processing: Malayalam
Found 2 artifacts


Processing text files:   0%|          | 0/2 [00:00<?, ?file/s, USX]

Processing text files: 100%|██████████| 2/2 [00:00<00:00,  2.39file/s, PARATEXT (USFM)]

✓ Successfully processed Malayalam

All languages processed!


# Alignment

## Using timing files

In [14]:
cmd = f"""
python utils/process_all_books_with_timing.py \
    --base_path "data/audios/{LANGUAGE}" \
    --timing_folder "data/audios/{LANGUAGE}/Timing Files/Timing Files Bundle" \
    --usfm_folder "data/texts/{LANGUAGE}/Paratext (USFM)/release/USX_1" \
    --workers 8
"""
print(cmd)


python utils/process_all_books_with_timing.py     --base_path "data/audios/Malayalam"     --timing_folder "data/audios/Malayalam/Timing Files/Timing Files Bundle"     --usfm_folder "data/texts/Malayalam/Paratext (USFM)/release/USX_1"     --workers 8



In [16]:
! python utils/process_all_books_with_timing.py     --base_path "data/audios/Malayalam"     --timing_folder "data/audios/Malayalam/Timing Files/Timing Files Bundle"     --usfm_folder "data/texts/Malayalam/PARATEXT (USFM)/release/USX_1"     --workers 8


Processing: data/audios/Malayalam
DataFrame saved to data/audios/Malayalam/books_to_process.csv

=== Books to Process ===
Total books: 66
USFM files found: 66
USFM files missing: 0

=== DataFrame Preview ===
          book_name book_code            testament  usfm_exists
0     1 Corinthians       1CO  New Testament - mp3         True
1            1 John       1JN  New Testament - mp3         True
2           1 Peter       1PE  New Testament - mp3         True
3   1 Thessalonians       1TH  New Testament - mp3         True
4         1 Timothy       1TI  New Testament - mp3         True
5     2 Corinthians       2CO  New Testament - mp3         True
6            2 John       2JN  New Testament - mp3         True
7           2 Peter       2PE  New Testament - mp3         True
8   2 Thessalonians       2TH  New Testament - mp3         True
9         2 Timothy       2TI  New Testament - mp3         True
10           3 John       3JN  New Testament - mp3         True
11             Acts    

## Using force alignment

In [17]:
cmd = f"""
python utils/process_all_books_force_align.py \
    --base_path "data/audios/{LANGUAGE}" \
    --usfm_folder "data/texts/{LANGUAGE}/Paratext (USFM)/release/USX_1" \
    --language "und" \
    --workers 4 \
    --chapter-intro "chapter introduction"
"""
print(cmd)


python utils/process_all_books_force_align.py     --base_path "data/audios/Polish"     --usfm_folder "data/texts/Polish/Paratext (USFM)/release/USX_1"     --language "und"     --workers 4     --chapter-intro "chapter introduction"



# Check before uploading

In [17]:
# Book code (from filenames) to full book name mapping
BOOK_CODE_TO_NAME = {
    # Old Testament
    "GEN": "Genesis", "EXO": "Exodus", "LEV": "Leviticus", "NUM": "Numbers",
    "DEU": "Deuteronomy", "JOS": "Joshua", "JDG": "Judges", "RUT": "Ruth",
    "1SA": "1 Samuel", "2SA": "2 Samuel", "1KI": "1 Kings", "2KI": "2 Kings",
    "1CH": "1 Chronicles", "2CH": "2 Chronicles", "EZR": "Ezra", "NEH": "Nehemiah",
    "EST": "Esther", "JOB": "Job", "PSA": "Psalms", "PRO": "Proverbs",
    "ECC": "Ecclesiastes", "SNG": "Song of Songs", "ISA": "Isaiah", "JER": "Jeremiah",
    "LAM": "Lamentations", "EZK": "Ezekiel", "DAN": "Daniel", "HOS": "Hosea",
    "JOL": "Joel", "AMO": "Amos", "OBA": "Obadiah", "JON": "Jonah",
    "MIC": "Micah", "NAM": "Nahum", "HAB": "Habakkuk", "ZEP": "Zephaniah",
    "HAG": "Haggai", "ZEC": "Zechariah", "MAL": "Malachi",
    # New Testament
    "MAT": "Matthew", "MRK": "Mark", "LUK": "Luke", "JHN": "John",
    "ACT": "Acts", "ROM": "Romans", "1CO": "1 Corinthians", "2CO": "2 Corinthians",
    "GAL": "Galatians", "EPH": "Ephesians", "PHP": "Philippians", "COL": "Colossians",
    "1TH": "1 Thessalonians", "2TH": "2 Thessalonians", "1TI": "1 Timothy",
    "2TI": "2 Timothy", "TIT": "Titus", "PHM": "Philemon", "HEB": "Hebrews",
    "JAS": "James", "1PE": "1 Peter", "2PE": "2 Peter", "1JN": "1 John",
    "2JN": "2 John", "3JN": "3 John", "JUD": "Jude", "REV": "Revelation",
}

# Book to testament mapping (constant, defined outside function)
BOOK_TO_TESTAMENT = {
    # Old Testament
    "Genesis": "Old Testament",
    "Exodus": "Old Testament",
    "Leviticus": "Old Testament",
    "Numbers": "Old Testament",
    "Deuteronomy": "Old Testament",
    "Joshua": "Old Testament",
    "Judges": "Old Testament",
    "Ruth": "Old Testament",
    "1 Samuel": "Old Testament",
    "2 Samuel": "Old Testament",
    "1 Kings": "Old Testament",
    "2 Kings": "Old Testament",
    "1 Chronicles": "Old Testament",
    "2 Chronicles": "Old Testament",
    "Ezra": "Old Testament",
    "Nehemiah": "Old Testament",
    "Esther": "Old Testament",
    "Job": "Old Testament",
    "Psalms": "Old Testament",
    "Proverbs": "Old Testament",
    "Ecclesiastes": "Old Testament",
    "Song of Songs": "Old Testament",
    "Isaiah": "Old Testament",
    "Jeremiah": "Old Testament",
    "Lamentations": "Old Testament",
    "Ezekiel": "Old Testament",
    "Daniel": "Old Testament",
    "Hosea": "Old Testament",
    "Joel": "Old Testament",
    "Amos": "Old Testament",
    "Obadiah": "Old Testament",
    "Jonah": "Old Testament",
    "Micah": "Old Testament",
    "Nahum": "Old Testament",
    "Habakkuk": "Old Testament",
    "Zephaniah": "Old Testament",
    "Haggai": "Old Testament",
    "Zechariah": "Old Testament",
    "Malachi": "Old Testament",
    # New Testament
    "Matthew": "New Testament",
    "Mark": "New Testament",
    "Luke": "New Testament",
    "John": "New Testament",
    "Acts": "New Testament",
    "Romans": "New Testament",
    "1 Corinthians": "New Testament",
    "2 Corinthians": "New Testament",
    "Galatians": "New Testament",
    "Ephesians": "New Testament",
    "Philippians": "New Testament",
    "Colossians": "New Testament",
    "1 Thessalonians": "New Testament",
    "2 Thessalonians": "New Testament",
    "1 Timothy": "New Testament",
    "2 Timothy": "New Testament",
    "Titus": "New Testament",
    "Philemon": "New Testament",
    "Hebrews": "New Testament",
    "James": "New Testament",
    "1 Peter": "New Testament",
    "2 Peter": "New Testament",
    "1 John": "New Testament",
    "2 John": "New Testament",
    "3 John": "New Testament",
    "Jude": "New Testament",
    "Revelation": "New Testament",
}


def get_alignment_dataframe(language: str, base_dir: str = "data/audios") -> pd.DataFrame:
    """
    Load alignment data for a given language and return a DataFrame.
    
    Args:
        language: The language name (e.g., "Yoruba")
        base_dir: Base directory for audio files (default: "data/audios")
    
    Returns:
        DataFrame with columns: audio_file, text_file, text, book, chapter, 
                               verse, testament, duration_seconds
    """
    alignment_dir = os.path.join(base_dir, language, "Alignment")
    
    # Collect all audio files recursively (.wav and .mp3)
    audio_files = []
    for root, dirs, files in os.walk(alignment_dir):
        for file in files:
            if file.lower().endswith(('.wav', '.mp3')):
                audio_files.append(os.path.join(root, file))
    
    # Each audio file has a corresponding .txt file with the same name
    text_files = [os.path.splitext(x)[0] + ".txt" for x in audio_files]
    
    # Build initial dataframe with file paths
    df = pd.DataFrame({
        "audio_file": audio_files,
        "text_file": text_files,
    })
    
    # Helper to safely read text file contents
    def read_text_file(file_path):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read().strip()
        except Exception:
            return None
    
    # Read transcript text from each text file
    df["text"] = df["text_file"].apply(read_text_file)
    
    # Extract metadata from file path structure:
    # Format: .../Alignment/{book_folder}/{BOOKCODE_CHAPTER_Verse_VERSE}.wav
    filename_stem = df["audio_file"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
    df["book_code"] = filename_stem.apply(lambda x: x.split("_")[0].upper())
    df["chapter"] = filename_stem.apply(lambda x: x.split("_")[1])
    df["verse"] = filename_stem.apply(lambda x: x.split("_")[-1])

    # Resolve full book name: use code->name mapping, fall back to folder name
    # (folder name is already a full name for previously processed languages like Marathi)
    book_folder = df["audio_file"].apply(lambda x: x.split("/")[-2])
    df["book"] = df["book_code"].map(BOOK_CODE_TO_NAME).fillna(book_folder)

    # Map book name to testament (Old/New)
    df["testament"] = df["book"].map(BOOK_TO_TESTAMENT)
    
    # Get audio duration in seconds
    df["duration_seconds"] = df["audio_file"].apply(audio_stats.get_audio_duration)

    # Reorder columns (drop book_code, it was only needed for name resolution)
    df = df[["audio_file", "text", "testament", "book", "chapter", "verse", "duration_seconds"]]
    
    return df

In [19]:
base_dir = "data/audios"
alignment_df = get_alignment_dataframe(LANGUAGE, base_dir)
alignment_df

,audio_file,text,testament,book,chapter,verse,duration_seconds
0,data/audios/Malayalam/Alignment/1 Peter/1PE_00...,"യേശുക്രിസ്തുവിന്റെ അപ്പൊസ്തലനായ പത്രോസ്, പൊന്ത...",New Testament,1 Peter,001,001,13.05
1,data/audios/Malayalam/Alignment/1 Peter/1PE_00...,പിതാവായ ദൈവത്തിന്റെ പൂർവജ്ഞാനത്തിന് അനുസൃതമായി...,New Testament,1 Peter,001,002,23.87
2,data/audios/Malayalam/Alignment/1 Peter/1PE_00...,നമ്മുടെ കർത്താവായ യേശുക്രിസ്തുവിന്റെ പിതാവായ ദ...,New Testament,1 Peter,001,003,13.80
3,data/audios/Malayalam/Alignment/1 Peter/1PE_00...,"ഈ പ്രത്യാശ, അനശ്വരവും നിർമലവും പ്രഭ മങ്ങാത്തതു...",New Testament,1 Peter,001,004,10.10
4,data/audios/Malayalam/Alignment/1 Peter/1PE_00...,"അങ്ങനെ, അന്ത്യകാലത്തു വെളിപ്പെടാൻ സജ്ജമാക്കിയി...",New Testament,1 Peter,001,005,7.68
...,...,...,...,...,...,...,...
30933,data/audios/Malayalam/Alignment/Zephaniah/ZEP_...,"ആ ദിവസത്തിൽ അവർ ജെറുശലേമിനോടു പറയും: “സീയോനേ, ...",Old Testament,Zephaniah,003,016,6.97
30934,data/audios/Malayalam/Alignment/Zephaniah/ZEP_...,"നിന്റെ ദൈവമായ യഹോവ നിന്നോടുകൂടെയുണ്ട്, അവിടന്ന...",Old Testament,Zephaniah,003,017,13.91
30935,data/audios/Malayalam/Alignment/Zephaniah/ZEP_...,“നിർദിഷ്ട പെരുന്നാളുകൾ നഷ്ടമായത് നിങ്ങൾക്കൊരു ...,Old Testament,Zephaniah,003,018,7.83
30936,data/audios/Malayalam/Alignment/Zephaniah/ZEP_...,നിന്നെ പീഡിപ്പിച്ച സകലരോടും ആ കാലത്ത് ഞാൻ ഇടപെ...,Old Testament,Zephaniah,003,019,11.90


In [22]:
from utils.hf_preprocessing import prepare_alignment_dataset

# Step 2: Convert the DataFrame to an HF Dataset, casting the audio
# column to Audio() and removing statistical outliers.
ds = prepare_alignment_dataset(alignment_df)

👀 ─ Found 30938 <text, audio> pairs
 · Checking if audio is readable...
😊 Found no unreadable audio files
 · Reading audio duration...
👀 ─ Found a total of 91.25 hours of readable data
 · Get transcript length...
 · Get num feature vectors...
👀 ┬ Found 62 audio clips over 30.0 seconds long
   └ Marking 3.33 hours of data as TOO_LONG
👀 ┬ Found 160 transcripts under 10 characters long
   └ Marking 0.79 hours of data as TOO_SHORT_TRANS
 · Get ratio (num_feats / transcript_len)...
👀 ┬ Found 51 <text, audio> pairs with more text than audio (bad for CTC)
   └ Marking 0.01 hours of data as OFFENDING_DATA
 · Calculating ratio (audio_len : transcript_len)...
👀 ┬ Found 475 <text, audio> pairs more than 3.0 standard deviations from the mean
   └ Marking 1.00 hours of data as NON_NORMAL
🎉 ┬ 30190 samples (86.12 hours) labeled as BEST
   └ Label distribution:
      - BEST: 30190
      - NON_NORMAL: 475
      - TOO_SHORT_TRANS: 160
      - TOO_LONG: 62
      - OFFENDING_DATA: 51


In [23]:
sample = alignment_df.iloc[20]

print(f"Book:    {sample['book']}  |  Chapter: {sample['chapter']}  |  Verse: {sample['verse']}")
print(f"Testament: {sample['testament']}")
print(f"Duration: {sample['duration_seconds']:.2f}s")
print(f"\nTranscription:\n{sample['text']}")
display(IPythonAudio(sample["audio_file"]))

Book:    1 Peter  |  Chapter: 001  |  Verse: 021
Testament: New Testament
Duration: 13.50s

Transcription:
ക്രിസ്തുവിനെ മരിച്ചവരിൽനിന്ന് ഉയിർപ്പിക്കുകയും തേജസ്കരിക്കുകയുംചെയ്ത ദൈവത്തിൽ, ക്രിസ്തു മുഖാന്തരം നിങ്ങൾ വിശ്വസിക്കുന്നു. അങ്ങനെ നിങ്ങളുടെ വിശ്വാസവും പ്രത്യാശയും ദൈവത്തിൽ ആയിരിക്കുകയും ചെയ്യുന്നു.


In [14]:
sample = alignment_df.iloc[20]

print(f"Book:    {sample['book']}  |  Chapter: {sample['chapter']}  |  Verse: {sample['verse']}")
print(f"Testament: {sample['testament']}")
print(f"Duration: {sample['duration_seconds']:.2f}s")
print(f"\nTranscription:\n{sample['text']}")
display(IPythonAudio(sample["audio_file"]))

Book:    1 John  |  Chapter: 002  |  Verse: 011
Testament: New Testament
Duration: 14.68s

Transcription:
എന്നാൽ സഹോദരനെ വെറുക്കുന്നവനോ ഇരുട്ടിൽ ഇരിക്കുന്നു; ഇരുട്ടിൽ നടക്കുകയും ചെയ്യുന്നു. ഇരുട്ട് അവന്‍റെ കണ്ണ് കുരുടാക്കുകയാൽ എവിടേക്ക് പോകുന്നു എന്നു അവൻ അറിയുന്നില്ല. ക്രിസ്തു-പാപവിമോചകൻ


In [24]:
# Remove the audio column
dataset_no_audio = ds.remove_columns(["audio"]).to_pandas()

def make_filename(row) -> str:
    return (
        f"{row['testament']}-{row['book']}-"
        f"{row['chapter']}-{row['verse']}.wav"
    )

dataset_no_audio["filename"] = dataset_no_audio.apply(make_filename, axis=1)

# Convert to pandas and save to CSV
dataset_no_audio.to_csv("Biblica_Malayalam.csv", index=False)

In [ ]:
# import os
# import shutil
# from tqdm import tqdm

# languages = os.listdir("data/audios")
# alignment_dirs = [os.path.join("data/audios", lang, "Alignment") for lang in languages]

# for dir_path in tqdm(alignment_dirs, desc="Deleting alignment directories"):
#     if os.path.isdir(dir_path):
#         shutil.rmtree(dir_path)

# print(f"Deleted {len(alignment_dirs)} alignment directories.")


Deleting alignment directories:   0%|          | 0/45 [00:00<?, ?it/s]

Deleting alignment directories: 100%|██████████| 45/45 [09:03<00:00, 12.07s/it]

Deleted 45 alignment directories.


In [3]:
texts_base = Path("data/texts")
paratext_release = Path("Paratext (USFM)") / "release"

languages = sorted(os.listdir(texts_base))
rows = []
for lang in languages:
    release_dir = texts_base / lang / paratext_release
    if release_dir.is_dir():
        usx_dirs = sorted(p.name for p in release_dir.iterdir() if p.is_dir())
    else:
        usx_dirs = []
    rows.append({"languages": lang, "USX": usx_dirs})

usx_release_df = pd.DataFrame(rows)
usx_release_df

,languages,USX
0,Apali,[USX_1]
1,Arabic Standard,[USX_1]
2,Assamese,[USX_1]
3,Bengali,[USX_1]
4,Central Kurdish,[USX_1]
5,Chhattisgarhi,[USX_1]
6,Chichewa,[USX_1]
7,Dawro,[USX_1]
8,Dholuo,[USX_1]
9,East Slovak Romani - Romani Carpathian,[]
